# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id`.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
metadata = dataset.metadata.to_json()
print(f"Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Keywords: {metadata.get('keywords', [])}")
print(f"License: {metadata.get('license')}")
print(f"Version: {metadata.get('version', 'N/A')}")

## 2. Data Overview

List available record sets, fields, and columns by their `@id`.

The Croissant API indexes all data entities with their unique `@id`.

In [ ]:
# List all Record Sets and their @id's
recordset_ids = [rs['@id'] for rs in dataset.metadata['recordSet']]
print('Record sets in the dataset:')
for rs in dataset.metadata['recordSet']:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '')}")

# For each record set, print its fields and columns by @id
for rs in dataset.metadata['recordSet']:
    print(f"\nRecordSet @id: {rs['@id']} | name: {rs.get('name','')}")
    # Fields
    fields = rs.get('field', [])
    if fields:
        print('  Fields:')
        for field in fields:
            # field can be dict or reference
            fobj = field if isinstance(field, dict) else next(f for f in dataset.metadata['field'] if f['@id'] == field)
            print(f"    - @id: {fobj['@id']} | name: {fobj.get('name','')}")
            # List columns for field
            columns = fobj.get('column', [])
            if columns:
                print('      Columns:')
                for col in columns:
                    cobj = col if isinstance(col, dict) else next(c for c in dataset.metadata['column'] if c['@id']==col)
                    print(f"        - @id: {cobj['@id']} | name: {cobj.get('name','')}")

## 3. Data Extraction

Load records from a selected record set by its `@id`. The list of record sets and fields is shown above. 

For this example, we'll extract the main record set. Please update the variable if needed.

In [ ]:
# Set the record set @id (use the one found in the overview; update as needed)
# For illustration, we'll select the first record set in the metadata
recordset_id = recordset_ids[0]

print(f"Selected record set: {recordset_id}")

# Extract all the records as a DataFrame
records = list(dataset.records(record_set=recordset_id))
df = pd.DataFrame(records)

print(f"Columns in the DataFrame (these are field @id's):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's perform basic EDA operations such as filtering and normalization of a numeric field, and grouping records by a key entity. All field names used below are their `@id` values for full traceability.

In [ ]:
# Inspect numeric fields (select by @id)
numeric_field_id = None
for c in df.columns:
    # Try to infer a numeric field by dtype
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_field_id = c
        break

if numeric_field_id is not None:
    print(f"Numeric field selected: {numeric_field_id}")
else:
    print("No numeric field found.")

# Set a threshold and filter
if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df["normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized field '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, "normalized"]].head())

    # Attempt grouping by another field (select the next available non-numeric column)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped data by '{group_field}':")
        print(grouped.head())
else:
    print("No numeric EDA possible on this record set.")

## 5. Visualization

Visualize the distribution of the chosen numeric field and its grouped means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        grouped.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion

In this notebook, we loaded, inspected, and visualized data from the FAIR² mass spectrometry dataset using `mlcroissant`, referring to all entities by their `@id` for reproducibility and traceability.

- We listed the record sets and their fields and columns by `@id`.
- We extracted and previewed records from the main record set.
- Basic EDA and visualization were performed for numeric values, also referenced by `@id`.

For deeper insight, consult the [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and accompanying documentation.